In [7]:
from agent.llm import call_llm
import json
from state.schema import InterviewState , CandidateLiveSignal , SessionMeta

In [ ]:
def interview_agent(resume_text,JD_text,company_info,candidate_profile): #these arguments will come from a different LLM call which we are calling profile extraction agent

    conversation_history = []

    tools = [
        {
            "type" : "function",
            "function" : {
                "name" : "company_search_tool",
                "description" : "Given the JD of a company, the tool can be used to fetch further details about the company for which the candidiate is applying",
                "parameters" : {
                    "type" : "object",
                    "properties" : {
                        "company_name" : {
                            "type" : "string",
                            "description" : "Name of the company for which the candidate is applying"
                        }
                    },
                    "required" : ["company_name"]
                }
            }
        }
        
    ]

    function_map = { "company_search_tool" : company_search}
    turns = 0
    state = StateSchema(
        resume_profile = ResumeProfile()
        interview_state = InterviewState()
        candidate_live_signal = CandidateLiveSignal()
        session_meta = SessionMeta()
    )
    while turns <= 15 and not interview_complete:
        message = [
                {"role":"system","content":system_prompt},
                #{"role":"user","content": interview_state},
            # {"role":"user","content": candidate_live_signal}
                ] + conversation_history
            llm_response = call_llm(model = "gpt-4o", messages = message, tools = tools)
            while llm_response.tool_calls:
                tool_name = lm_response.tool_calls[0].function.name
                arguments = json_loads(lm_response.tool_calls[0].function.arguments)
                #But you're also missing the assistant message that contains the tool call itself before the tool result.
                tool_result = function_map[tool_name](**arguments)
                message.append({"role":"tool","content" : str(tool_result),"tool_call_id": llm_repsonse.tool_calls[0].id})
                llm_response = call_llm(model = "gpt-4o", messages = message, tools = tools)
            
            llm_question = llm_response.content
            if "INTERVIEW_COMPLETE" in llm_question:
                interview_complete = True
            else:
                turns += 1
                candidate_response = input("please enter your response: ")
                conversation_history.append({"role":"assistant","content":llm_question},{"role":"user","content":candidate_response})
                ##update the interview_state -> via code and via LLM
                ##update the candidate live signal -> via LLM
    ## call llm to genrate the report -> a diff system prompt will be used to generating the report

            


                
            
